### Excercise 1 -- Monte carlo Serial Implementation

In [4]:
import random
import statistics
import time
import math
from multiprocessing import Pool, cpu_count

def estimate_pi_serial(num_samples):

    hits = 0

    for i in range(num_samples):
        x = random.random()
        y = random.random()

        if x * x + y * y <= 1.0:
            hits += 1

    return 4.0 * hits / num_samples



if __name__ == "__main__":
    
    NUM_SAMPLES = 10000000
    TRUE_PI = math.pi

    times = []

    for i in range (3):
        start = time.perf_counter()
        pi_estimate = estimate_pi_serial(NUM_SAMPLES)
        end = time.perf_counter()

        result = times.append(end - start)
    
    time_serial = statistics.median(times)
    error_serial = abs(pi_estimate - TRUE_PI)
    print(f"Serial Estimate: {pi_estimate:.6f}, Time: {time_serial:.4f} seconds, Error: {error_serial:.6f}")

Serial Estimate: 3.141236, Time: 2.0202 seconds, Error: 0.000357


Serial Estimate: 3.141592, Time: 1.7396 seconds, Error: 0.000001
Serial Estimate: 3.141203, Time: 1.8428 seconds, Error: 0.000390
Serial Estimate: 3.141236, Time: 2.0202 seconds, Error: 0.000357

### Questions

####  How accurate is the estimate? Run several times — does it vary?
- The estimate is accurate to the 4th decimal.
- Yes, everything after the 4th decimal changes with each run.

####  What is the serial time? This will be your speedup denominator in E3.
- Median time: Time: 1.7396

---

### Excercise 2 -- Monte carlo parallel Implementation

In [ ]:
from multiprocessing import Pool
import os
import time
import math

def estimate_pi_chunk(num_samples):

    hits = 0

    for i in range(num_samples):
        x = random.random()
        y = random.random()
        if x * x + y * y <= 1.0:
            hits += 1
    return hits


def estimate_pi_parallel(num_samples, num_processes):
    
    samples_per_worker = num_samples // num_processes

    tasks = [samples_per_worker] * num_processes

    with Pool(processes=num_processes) as pool:
        results = pool.map(estimate_pi_chunk, tasks)

    total_hits = sum(results)
    total_samples = samples_per_worker * num_processes

    return 4.0 * total_hits / total_samples



def benchmark_parallel(num_samples):
    
    max_procs = os.cpu_count()

    for p in range(1, max_procs + 1):
        t0 = time.perf_counter()
        pi_est = estimate_pi_parallel(num_samples, p)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        speedup =  1.7396 / elapsed

        print(
            f"Procs: {p:2d} | π ≈ {pi_est:.6f} | "
            f"time = {elapsed:.4f} s | speedup = {speedup:.2f}x"
        )


if __name__ == "__main__":
    NUM_SAMPLES = 10000000
    benchmark_parallel(NUM_SAMPLES)

(nsc2026) C:\Users\braec\Desktop\mandelbrot_parallel>C:\Users\braec\miniforge3\envs\nsc2026\python.exe c:/Users/braec/Desktop/mandelbrot_parallel/excercises.py
| Procs | π Estimate | Time (s) | Speedup |
|------:|-----------:|---------:|--------:|
| 1  | 3.141176 | 1.8455 | 0.94x |
| 2  | 3.141962 | 0.9505 | 1.83x |
| 3  | 3.142066 | 0.7205 | 2.41x |
| 4  | 3.141993 | 0.6042 | 2.88x |
| 5  | 3.142090 | 0.5277 | 3.30x |
| 6  | 3.141932 | 0.5023 | 3.46x |
| 7  | 3.140851 | 0.4571 | 3.81x |
| 8  | 3.141017 | 0.4596 | 3.79x |
| 9  | 3.141896 | 0.4495 | 3.87x |
| 10 | 3.140967 | 0.4472 | 3.89x |
| 11 | 3.142178 | 0.4619 | 3.77x |
| 12 | 3.141489 | 0.4654 | 3.74x |
| 13 | 3.141977 | 0.4372 | 3.98x |
| 14 | 3.140786 | 0.4485 | 3.88x |
| 15 | 3.140800 | 0.4851 | 3.59x |
| 16 | 3.141876 | 0.4889 | 3.56x |

---

#### Questions

**Do all worker counts give the same pi hat Why or why not?**

No, they do not give exactly the same value. Each worker generates its own random set of points, so the Monte Carlo estimate varies slightly due to randomness.

---

**At which count do you first see a meaningful speedup?**

At around **2 processes**, where the speedup is approximately **2x**, which is clearly noticeable and meaningful.